<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Phishing_Risk_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python program that analyzes sample emails using multiple phishing indicators and assigns each email a Low, Medium, or High risk classification based on a combined risk score.

**Algorithm**

1. Load Emails:
Create a collection of sample emails containing sender, subject, body, and hyperlinks.

2. Analyze Sender:
Check for suspicious sender addresses, unusual domains, and domain spoofing patterns.

3. Analyze Subject:
Identify urgency, threats, account warnings, and suspicious requests.

4. Analyze Body:
Search for credential requests, urgent expressions, financial requests, and suspicious instructions.

5. Check Links:
Examine hyperlinks for suspicious domains, HTTP connections, IP addresses, and misleading URLs.

6. Calculate Score:
Assign points for each detected indicator and combine them into a total phishing-risk score.

7. Classify Risk:
Classify emails according to their total score:

0–3: Low Risk

4–7: Medium Risk

8+: High Risk

8. Display Results:
Display the detected indicators, risk score, classification, and reasons for the classification.



In [1]:
import re
from urllib.parse import urlparse

# ---------------------------------------------------------
# PHISHING EMAIL RISK CLASSIFIER
# ---------------------------------------------------------

emails = [
    {
        "id": "EMAIL-001",
        "sender": "support@company.com",
        "subject": "Monthly Newsletter",
        "body": "Hello, please find our monthly company newsletter.",
        "links": ["https://www.company.com/news"]
    },

    {
        "id": "EMAIL-002",
        "sender": "security-alert@gmail.com",
        "subject": "Urgent: Verify Your Account",
        "body": """
        Your account will be suspended immediately.
        Click the link and verify your password and username.
        """,
        "links": ["http://account-verification-login.xyz/verify"]
    },

    {
        "id": "EMAIL-003",
        "sender": "admin@paypa1-security.com",
        "subject": "Your account requires immediate action",
        "body": """
        We detected suspicious activity on your account.
        You must urgently confirm your password, username,
        credit card number and OTP to avoid account suspension.
        Failure to respond within 24 hours will result in closure.
        """,
        "links": [
            "http://paypa1-security.com/login",
            "http://192.168.1.50/verify"
        ]
    }
]


# ---------------------------------------------------------
# INDICATOR DEFINITIONS
# ---------------------------------------------------------

urgency_words = [
    "urgent",
    "immediately",
    "act now",
    "within 24 hours",
    "suspended",
    "suspension",
    "limited time",
    "failure"
]

credential_words = [
    "password",
    "username",
    "login",
    "credential",
    "otp",
    "pin",
    "verify your account"
]

financial_words = [
    "credit card",
    "bank account",
    "payment",
    "transaction",
    "money"
]

suspicious_domain_words = [
    "login",
    "verify",
    "security",
    "secure",
    "account",
    "update"
]

common_domains = [
    "gmail.com",
    "outlook.com",
    "yahoo.com",
    "company.com"
]


# ---------------------------------------------------------
# ANALYZE SENDER
# ---------------------------------------------------------

def analyze_sender(sender):

    indicators = []
    score = 0

    domain = sender.split("@")[-1].lower()

    # Free email provider used for security-type message
    if domain in ["gmail.com", "yahoo.com", "outlook.com"]:
        indicators.append(
            "Sender uses a free email provider"
        )
        score += 1

    # Suspicious characters / numbers
    if re.search(r"[0-9]", domain):

        indicators.append(
            "Sender domain contains unusual numbers"
        )
        score += 2

    # Suspicious security-like domain
    if any(word in domain for word in suspicious_domain_words):

        indicators.append(
            "Sender domain contains security/account-related terms"
        )
        score += 2

    return score, indicators


# ---------------------------------------------------------
# ANALYZE TEXT
# ---------------------------------------------------------

def analyze_text(subject, body):

    indicators = []
    score = 0

    text = (subject + " " + body).lower()

    # Urgency
    urgency_found = []

    for word in urgency_words:
        if word in text:
            urgency_found.append(word)

    if urgency_found:

        indicators.append(
            "Urgency expressions detected: "
            + ", ".join(urgency_found)
        )

        score += min(len(urgency_found), 3)

    # Credential requests
    credential_found = []

    for word in credential_words:
        if word in text:
            credential_found.append(word)

    if credential_found:

        indicators.append(
            "Credential-related request detected: "
            + ", ".join(credential_found)
        )

        score += 3

    # Financial information
    financial_found = []

    for word in financial_words:
        if word in text:
            financial_found.append(word)

    if financial_found:

        indicators.append(
            "Financial information mentioned: "
            + ", ".join(financial_found)
        )

        score += 2

    # Suspicious subject
    subject_lower = subject.lower()

    if any(word in subject_lower for word in urgency_words):

        indicators.append(
            "Suspicious or urgent subject line"
        )

        score += 2

    return score, indicators


# ---------------------------------------------------------
# ANALYZE HYPERLINKS
# ---------------------------------------------------------

def analyze_links(links):

    indicators = []
    score = 0

    for link in links:

        parsed = urlparse(link)

        domain = parsed.netloc.lower()

        # HTTP instead of HTTPS
        if parsed.scheme.lower() == "http":

            indicators.append(
                f"Unencrypted HTTP link: {link}"
            )

            score += 1

        # Direct IP address
        if re.match(r"^\d+\.\d+\.\d+\.\d+", domain):

            indicators.append(
                f"Link uses an IP address: {link}"
            )

            score += 3

        # Suspicious domain
        if any(word in domain for word in suspicious_domain_words):

            indicators.append(
                f"Suspicious domain pattern: {domain}"
            )

            score += 2

        # Hyphenated or unusual domain
        if "-" in domain:

            indicators.append(
                f"Unusual hyphenated domain: {domain}"
            )

            score += 1

        # Unknown domain
        if not any(domain.endswith(d) for d in common_domains):

            indicators.append(
                f"Domain is not in the trusted-domain list: {domain}"
            )

            score += 1

    return score, indicators


# ---------------------------------------------------------
# RISK CLASSIFICATION
# ---------------------------------------------------------

def classify(score):

    if score >= 8:
        return "HIGH RISK"

    elif score >= 4:
        return "MEDIUM RISK"

    else:
        return "LOW RISK"


# ---------------------------------------------------------
# PROCESS EMAILS
# ---------------------------------------------------------

for email in emails:

    total_score = 0
    all_indicators = []

    sender_score, sender_indicators = analyze_sender(
        email["sender"]
    )

    text_score, text_indicators = analyze_text(
        email["subject"],
        email["body"]
    )

    link_score, link_indicators = analyze_links(
        email["links"]
    )

    total_score = (
        sender_score +
        text_score +
        link_score
    )

    all_indicators.extend(sender_indicators)
    all_indicators.extend(text_indicators)
    all_indicators.extend(link_indicators)

    risk = classify(total_score)

    # -----------------------------------------------------
    # DISPLAY RESULT
    # -----------------------------------------------------

    print("\n" + "=" * 70)
    print("EMAIL:", email["id"])
    print("=" * 70)

    print("Sender :", email["sender"])
    print("Subject:", email["subject"])

    print("\nDetected Indicators:")

    if all_indicators:

        for number, indicator in enumerate(
            all_indicators, 1
        ):
            print(f"{number}. {indicator}")

    else:
        print("No significant phishing indicators detected.")

    print("\nRisk Score :", total_score)
    print("Classification:", risk)

    print("\nWhy Classified:")

    if risk == "HIGH RISK":

        print(
            "Multiple phishing indicators were detected, "
            "including suspicious links, credential requests, "
            "urgency, or suspicious sender information."
        )

    elif risk == "MEDIUM RISK":

        print(
            "Several suspicious indicators were detected, "
            "but the overall score does not reach the high-risk threshold."
        )

    else:

        print(
            "Few or no suspicious indicators were detected "
            "and the overall risk score is low."
        )


print("\n" + "=" * 70)
print("PHISHING ANALYSIS COMPLETE")
print("=" * 70)


EMAIL: EMAIL-001
Sender : support@company.com
Subject: Monthly Newsletter

Detected Indicators:
No significant phishing indicators detected.

Risk Score : 0
Classification: LOW RISK

Why Classified:
Few or no suspicious indicators were detected and the overall risk score is low.

EMAIL: EMAIL-002
Sender : security-alert@gmail.com
Subject: Urgent: Verify Your Account

Detected Indicators:
1. Sender uses a free email provider
2. Urgency expressions detected: urgent, immediately, suspended
3. Credential-related request detected: password, username, verify your account
4. Suspicious or urgent subject line
5. Unencrypted HTTP link: http://account-verification-login.xyz/verify
6. Suspicious domain pattern: account-verification-login.xyz
7. Unusual hyphenated domain: account-verification-login.xyz
8. Domain is not in the trusted-domain list: account-verification-login.xyz

Risk Score : 14
Classification: HIGH RISK

Why Classified:
Multiple phishing indicators were detected, including suspici

**Result**

The Python program successfully analyzed a collection of sample emails using multiple phishing indicators rather than relying on a single keyword. It examined sender addresses, subject lines, message bodies, hyperlinks, urgency expressions, credential requests, and suspicious domains. Each email was assigned a risk score and Low, Medium, or High